# Jev CarRacing v2 — corrected perception and explicit assistance

[Open in Colab](https://colab.research.google.com/github/vtavakkoli/simple-jev/blob/main/notebooks/Jev_CarRacing_Colab.ipynb)

Default `CONTROL_MODE="assisted"`: a pixel lane follower steers every physics frame; Jev chooses small steering and speed adjustments. This is a hybrid controller, not evidence that Jev alone learned to race. `CONTROL_MODE="jev_direct"` retains direct Jev driving for comparison, without the lane follower.

Both modes wait for the startup camera zoom, merge road segments split by the car sprite, use the selected categorical action instead of averaging gas and brake, and read actual simulator speed (explicit privileged telemetry). Neither controller reads the track waypoints. The speed reading is **not** a pixel-only estimate.

Store `TYPESAFE_API_KEY` in Colab Secrets. Offline validation does not call Jev. Live Jev performance remains unverified.


In [ ]:
#@title 1. Install dependencies
!pip -q install "gymnasium[box2d]==1.3.0" imageio imageio-ffmpeg requests pillow matplotlib


In [ ]:
#@title 2. Imports and configuration
import os, time, json, getpass
from collections import deque
from pathlib import Path
import numpy as np
import requests
import gymnasium as gym
import imageio.v2 as imageio
import matplotlib.pyplot as plt
from PIL import Image as PILImage
from IPython.display import Video, Image, display

API_URL = "https://api.typesafe.ai/v1/systemone"
MODEL = "jev-latest"
ENV_ID = "CarRacing-v3"
SEED = 0
MAX_STEPS = 3000
CONTROL_MODE = "assisted" # "assisted" or "jev_direct"
CAMERA_WARMUP = 50
MIN_CONFIDENCE = 0.40
STALL_FRAMES = 500
CONTROL_HORIZON = 10

MAX_STEER_DELTA_PER_FRAME = 0.35
MAX_LONGITUDINAL_DELTA_PER_FRAME = 0.30
HISTORY_LEN = 10

VIDEO_EVERY = 2
GIF_EVERY = 5
OUTPUT_DIR = Path("/content") if Path("/content").exists() else Path.cwd()
VIDEO_PATH = str(OUTPUT_DIR / "jev_car_racing_v2.mp4")
GIF_PATH = str(OUTPUT_DIR / "jev_car_racing_v2.gif")
LOG_PATH = str(OUTPUT_DIR / "jev_car_racing_v2.json")

REQUEST_TIMEOUT_S = 20
MAX_RETRIES = 3
SCAN_ROWS = [68, 62, 56, 50, 44, 38]

def load_typesafe_key():
    key = None
    try:
        from google.colab import userdata
        key = userdata.get("TYPESAFE_API_KEY")
    except Exception:
        pass
    if not key:
        key = os.environ.get("TYPESAFE_API_KEY")
    if not key:
        key = getpass.getpass("TypeSafe API key from https://typesafe.ai: ").strip()
    if not key:
        raise RuntimeError("No TYPESAFE_API_KEY supplied.")
    return key

TYPESAFE_API_KEY = load_typesafe_key()
print("✓ API key loaded")
print("Environment:", ENV_ID, "| model:", MODEL)

In [ ]:
#@title 3. Road perception and explicit simulator speed

def road_mask_from_rgb(frame):
    img = np.asarray(frame, dtype=np.float32)
    mask = (np.ptp(img, axis=2) < 20) & (img.mean(axis=2) > 65) & (img.mean(axis=2) < 165)
    mask[82:] = False
    return mask

def scan_road(mask):
    previous_center = 48.0
    centers, widths, valid = [], [], []
    for y in SCAN_ROWS:
        xs = np.flatnonzero(mask[y])
        # Bridge narrow holes from the car sprite and lane markings.
        groups = np.split(xs, np.where(np.diff(xs) > 8)[0] + 1) if len(xs) else []
        groups = [g for g in groups if len(g) >= 3]
        if not groups:
            centers.append(None); widths.append(0.0); valid.append(False)
            continue
        g = min(groups, key=lambda g: abs((g[0]+g[-1])/2 - previous_center))
        previous_center = float((g[0]+g[-1])/2)
        centers.append((previous_center - 48.0)/48.0)
        widths.append(float(g[-1]-g[0]+1)/96.0)
        valid.append(True)
    return centers, widths, valid

def extract_visual_state(frame, speed):
    centers, widths, valid = scan_road(road_mask_from_rgb(frame))
    rows = [(y, c) for y, c, ok in zip(SCAN_ROWS, centers, valid) if ok]
    return {
        "road_center_offsets_near_to_far": centers,
        "road_width_fraction_near_to_far": widths,
        "scanline_valid": valid,
        "road_visibility": float(np.mean(valid)),
        "near_center_offset": rows[0][1] if rows else None,
        "far_center_offset": rows[-1][1] if rows else None,
        "speed": float(speed),
        "speed_source": "simulator hull velocity magnitude; not estimated from pixels",
    }

def read_speed(env):
    return float(np.linalg.norm(env.unwrapped.car.hull.linearVelocity))

def perception_overlay(frame):
    out = np.asarray(frame).copy()
    mask = road_mask_from_rgb(frame)
    out[mask] = (0.6*out[mask] + 0.4*np.array([255,255,0])).astype(np.uint8)
    centers, _, valid = scan_road(mask)
    for y, c, ok in zip(SCAN_ROWS, centers, valid):
        if ok:
            x = int(np.clip(round(48 + 48*c), 0, 95))
            out[y-1:y+2, max(0,x-1):min(96,x+2)] = [255,0,255]
    return out

def reset_race(env, seed):
    obs, info = env.reset(seed=seed)
    warmup_reward = 0.0
    for _ in range(CAMERA_WARMUP):
        obs, reward, terminated, truncated, info = env.step(np.zeros(3, dtype=np.float32))
        warmup_reward += float(reward)
        if terminated or truncated:
            raise RuntimeError("Environment ended during stationary camera warmup")
    return obs, info, warmup_reward


In [ ]:
#@title 4. Explicit hybrid lane follower and direct-action alternatives

def to_gym_action(control):
    steer, longitudinal = np.clip(np.asarray(control, dtype=np.float32), -1, 1)
    return np.array([steer, max(float(longitudinal), 0), max(-float(longitudinal), 0)], dtype=np.float32)

# In assisted mode Jev selects a residual steering trim and speed multiplier.
ASSISTED = {
    f"{direction}_{pace}": np.array([trim, factor], dtype=np.float32)
    for direction, trim in [("left", -.03), ("center", 0.), ("right", .03)]
    for pace, factor in [("slow", .95), ("normal", 1.), ("fast", 1.05)]
}
DIRECT = {
    f"steer_{i}_drive_{j}": np.array([steer, longitudinal], dtype=np.float32)
    for i, steer in enumerate([-.6, -.3, -.1, 0., .1, .3, .6])
    for j, longitudinal in enumerate([-.3, 0., .15, .3])
}
assert CONTROL_MODE in ("assisted", "jev_direct")
CANDIDATES = ASSISTED if CONTROL_MODE == "assisted" else DIRECT
CRITERIA = {
    key: (f"Lane-follower steering trim {v[0]:+.2f}; target speed multiplier {v[1]:.2f}. Preserve lane following."
          if CONTROL_MODE == "assisted" else f"Steering {v[0]:+.2f} (negative left, positive right); longitudinal {v[1]:+.2f} (positive gas, negative brake).")
    for key, v in CANDIDATES.items()
}

def assisted_control(vision, choice="center_normal"):
    rows = [(y, c) for y, c, ok in zip(SCAN_ROWS, vision["road_center_offsets_near_to_far"], vision["scanline_valid"]) if ok]
    if not rows:
        # Do not blindly accelerate after losing the road. The stall watchdog will stop the run.
        return np.array([0., -.15 if vision["speed"] > 3 else 0.], dtype=np.float32)
    y, c = min(rows, key=lambda row: abs(row[0]-50))
    angle = float(np.arctan2(48*c, 72-y))
    trim, factor = ASSISTED.get(choice, ASSISTED["center_normal"])
    steer = float(np.clip(1.4*angle + trim, -1, 1))
    target_speed = (22 - 10*min(abs(angle), 1))*factor
    error = target_speed - vision["speed"]
    longitudinal = float(np.clip(.06*error, 0, .3) - np.clip(-.03*error, 0, .2))
    return np.array([steer, longitudinal], dtype=np.float32)


In [ ]:
#@title 5. Jev racing policy
class JevCarRacingPolicy:
    def __init__(self, api_key):
        self.session = requests.Session()
        self.session.headers.update({
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json",
        })
        self.history = deque(maxlen=HISTORY_LEN)
        self.current_control = np.array([0.0,0.0], dtype=np.float32)

    def state(self, vision, episode_return, step):
        history = []
        for h in self.history:
            history.append({
                "control": [round(float(x),3) for x in h["control"]],
                "reward_sum": round(float(h["reward_sum"]),3),
                "terminated": bool(h["terminated"]),
                "road_visibility_after": round(float(h["road_visibility_after"]),3),
            })

        return {
            "environment": "Gymnasium CarRacing-v3",
            "objective": (
                "Maximize cumulative reward by following the road and visiting new road tiles. "
                "Avoid leaving the road/playfield. Use throttle on straights and reduce speed for turns."
            ),
            "action_semantics": {
                "steering": "-1 full left, 0 straight, +1 full right",
                "longitudinal": "+1 full gas, 0 coast, -1 strong brake",
            },
            "vision_semantics": {
                "offsets": "negative = road left of image center, positive = road right; ordered near-to-far",
                "missing": "null means no detected road; never interpret missing road as centered",
            },
            "controller_mode": CONTROL_MODE,
            "assistance": "Pixel lane follower; choices are bounded trim and pace" if CONTROL_MODE == "assisted" else "Direct motor actions",
            "step": int(step),
            "episode_return": round(float(episode_return),3),
            "vision": vision,
            "current_control": {
                "steering": round(float(self.current_control[0]),3),
                "longitudinal": round(float(self.current_control[1]),3),
            },
            "recent_action_outcomes": history,
        }

    def call(self, state):
        payload = {
            "model": MODEL,
            "state": state,
            "questions": {
                "driving_action": {
                    "type": "choice",
                    "instructions": (
                        "Choose the action appropriate to controller_mode. In assisted mode prefer center_normal; use small trims only for persistent offset. "
                        "Use near-to-far road center offsets to infer the upcoming turn. "
                        "Use road width, visibility, measured speed and recent reward outcomes. At low speed on visible road use gas, not brake. "
                        "Avoid excessive throttle in strong turns and react when road geometry changes."
                    ),
                    "criteria": CRITERIA,
                }
            },
        }

        last_error = None
        for attempt in range(MAX_RETRIES):
            t0 = time.perf_counter()
            try:
                r = self.session.post(API_URL, json=payload, timeout=REQUEST_TIMEOUT_S)
                latency = (time.perf_counter()-t0)*1000.0

                if r.status_code in (429,529) or 500 <= r.status_code < 600:
                    last_error = RuntimeError(f"HTTP {r.status_code}: {r.text[:300]}")
                    time.sleep(min(2**attempt,4))
                    continue

                r.raise_for_status()
                data = r.json()
                a = data["answers"]["driving_action"]

                return {
                    "choice": a["choice"],
                    "confidence": float(a.get("confidence",0.0)),
                    "probabilities": a.get("probabilities",{}),
                    "latency_ms": latency,
                }
            except Exception as e:
                last_error = e
                if attempt + 1 < MAX_RETRIES:
                    time.sleep(min(2**attempt,4))

        raise RuntimeError(f"Jev request failed: {last_error}")

    def probabilities_to_control(self, d):
        winner = d["choice"]
        if winner not in CANDIDATES or not np.isfinite(d["confidence"]) or not 0 <= d["confidence"] <= 1:
            raise ValueError("Invalid Jev choice or confidence")
        # Choice probabilities describe alternatives; do not average incompatible actions.
        top = [(key, float(value)) for key, value in d.get("probabilities", {}).items()
               if key in CANDIDATES and np.isfinite(float(value)) and float(value) >= 0]
        return CANDIDATES[winner].copy(), winner, sorted(top, key=lambda x:x[1], reverse=True)[:5]

    def decide(self, vision, episode_return, step):
        d = self.call(self.state(vision, episode_return, step))
        target, winner, top = self.probabilities_to_control(d)
        return {**d, "target_control":target, "winner":winner, "top_distribution":top}

    def record(self, control, rewards, terminated, vision_after):
        self.history.append({
            "control": np.asarray(control,dtype=np.float32).copy(),
            "reward_sum": float(np.sum(rewards)),
            "terminated": bool(terminated),
            "road_visibility_after": float(vision_after["road_visibility"]),
        })

print("✓ Jev policy ready")

In [ ]:
#@title 6. Check perception after camera warmup and test API
with gym.make(ENV_ID, render_mode="rgb_array", continuous=True, domain_randomize=False, max_episode_steps=MAX_STEPS) as env:
    obs, _, warmup_reward = reset_race(env, SEED)
    vision = extract_visual_state(obs, read_speed(env))
    print(json.dumps(vision, indent=2))
    plt.imshow(perception_overlay(obs)); plt.title("Road mask after startup zoom"); plt.axis("off"); plt.show()
    test_policy = JevCarRacingPolicy(TYPESAFE_API_KEY)
    try:
        test = test_policy.decide(vision, warmup_reward, CAMERA_WARMUP)
        print("Jev choice:", test["winner"], "confidence:", test["confidence"])
        print("Controller mode:", CONTROL_MODE)
    finally:
        test_policy.session.close()


In [ ]:
#@title 7. Run racing with explicit control-mode logging
assert MAX_STEPS > CAMERA_WARMUP
policy = JevCarRacingPolicy(TYPESAFE_API_KEY)
env = gym.make(ENV_ID, render_mode="rgb_array", continuous=True, domain_randomize=False, max_episode_steps=MAX_STEPS)
obs, _, episode_return = reset_race(env, SEED)
step = CAMERA_WARMUP
reward_history, steer_history, long_history = [], [], []
visibility_history, speed_history, confidence_history = [], [], []
decision_log, gif_frames = [], []
api_failures = low_confidence_fallbacks = 0
last_progress_step = step
visited = env.unwrapped.tile_visited_count
stop_reason = "frame_limit"
writer = imageio.get_writer(VIDEO_PATH, fps=50.0/VIDEO_EVERY, codec="libx264", pixelformat="yuv420p", macro_block_size=1)
try:
    while step < MAX_STEPS:
        vision = extract_visual_state(obs, read_speed(env))
        try:
            d = policy.decide(vision, episode_return, step)
            selected = d["winner"]
            if d["confidence"] < MIN_CONFIDENCE and CONTROL_MODE == "assisted":
                selected = "center_normal"
                low_confidence_fallbacks += 1
        except Exception as error:
            api_failures += 1
            selected = "center_normal" if CONTROL_MODE == "assisted" else "api_stop"
            d = dict(choice="api_error", confidence=0., latency_ms=None, error=str(error), top_distribution=[])
            print("API fallback:", selected, str(error))
        rewards = []
        for _ in range(min(CONTROL_HORIZON, MAX_STEPS-step)):
            vision = extract_visual_state(obs, read_speed(env))
            control = (assisted_control(vision, selected) if CONTROL_MODE == "assisted"
                       else DIRECT.get(selected, np.array([0., -.15], dtype=np.float32)).copy())
            # Direct actions may change abruptly; assisted steering is recomputed each physics frame.
            if CONTROL_MODE == "jev_direct":
                control = policy.current_control + np.clip(control-policy.current_control,
                    [-MAX_STEER_DELTA_PER_FRAME,-MAX_LONGITUDINAL_DELTA_PER_FRAME],
                    [MAX_STEER_DELTA_PER_FRAME,MAX_LONGITUDINAL_DELTA_PER_FRAME])
            obs, reward, terminated, truncated, _ = env.step(to_gym_action(control))
            policy.current_control = control.copy()
            episode_return += float(reward); rewards.append(float(reward)); step += 1
            after = extract_visual_state(obs, read_speed(env))
            reward_history.append(episode_return); steer_history.append(float(control[0])); long_history.append(float(control[1]))
            visibility_history.append(after["road_visibility"]); speed_history.append(after["speed"])
            if env.unwrapped.tile_visited_count > visited:
                visited = env.unwrapped.tile_visited_count; last_progress_step = step
            if step % VIDEO_EVERY == 0:
                writer.append_data(env.render())
            if step % GIF_EVERY == 0:
                gif_frames.append(np.asarray(PILImage.fromarray(env.render()).resize((480,384))))
            stalled = step-last_progress_step >= STALL_FRAMES
            if terminated or truncated or stalled:
                stop_reason = "terminated" if terminated else "time_limit" if truncated else "no_new_tiles"
                break
        policy.record(control, rewards, terminated or truncated, after)
        decision_log.append(dict(step=step, choice=d["choice"], applied_choice=selected,
            confidence=float(d["confidence"]), latency_ms=d["latency_ms"], error=d.get("error"),
            control=[float(x) for x in control], speed=after["speed"],
            return_total=episode_return, tiles=visited, vision=after))
        confidence_history.append(float(d["confidence"]))
        if len(decision_log) <= 10 or len(decision_log)%20 == 0:
            print(f"decision {len(decision_log):04d} | step {step:04d} | {CONTROL_MODE} | {selected} | "
                  f"return {episode_return:+.1f} | steer {control[0]:+.2f} | long {control[1]:+.2f} | "
                  f"speed {after['speed']:.1f} | tiles {visited}/{len(env.unwrapped.track)}", flush=True)
        if terminated or truncated or stalled:
            break
    coverage = visited/len(env.unwrapped.track)
finally:
    writer.close(); env.close(); policy.session.close()
if gif_frames:
    imageio.mimsave(GIF_PATH, gif_frames, duration=1000.0*GIF_EVERY/50, loop=0)
summary = dict(controller_mode=CONTROL_MODE, seed=SEED, steps=step, total_reward=episode_return,
               road_coverage=coverage, stop_reason=stop_reason, api_failures=api_failures,
               low_confidence_fallbacks=low_confidence_fallbacks)
Path(LOG_PATH).write_text(json.dumps(dict(summary=summary, decisions=decision_log), indent=2, allow_nan=False))
print("\n=== RESULT ===\n" + json.dumps(summary, indent=2))
latencies = [d["latency_ms"] for d in decision_log if d["latency_ms"] is not None]
if latencies:
    print("Mean / p95 API latency (ms):", round(float(np.mean(latencies)),1), round(float(np.percentile(latencies,95)),1))


In [ ]:
#@title 8. Video / GIF / plots
print("MP4:")
display(Video(VIDEO_PATH, embed=True, html_attributes="controls loop"))

print("GIF fallback:")
display(Image(filename=GIF_PATH))

plt.figure(figsize=(12,4))
plt.plot(reward_history)
plt.xlabel("Gym step")
plt.ylabel("cumulative reward")
plt.title("Jev CarRacing reward")
plt.grid(True,alpha=.25)
plt.show()

plt.figure(figsize=(12,4))
plt.plot(steer_history,label="steering")
plt.plot(long_history,label="longitudinal (+gas / -brake)")
plt.ylim(-1.05,1.05)
plt.legend()
plt.grid(True,alpha=.25)
plt.show()

plt.figure(figsize=(12,4))
plt.plot(visibility_history,label="road visibility")
plt.plot(np.asarray(speed_history)/120.0,label="measured simulator speed / 120")
plt.legend()
plt.grid(True,alpha=.25)
plt.show()

plt.figure(figsize=(12,4))
plt.plot(confidence_history)
plt.ylim(-.02,1.02)
plt.title("Jev action confidence")
plt.grid(True,alpha=.25)
plt.show()

print("MP4:",VIDEO_PATH)
print("GIF:",GIF_PATH)
print("log:",LOG_PATH)

In [ ]:
#@title 9. Optional offline assisted-controller benchmark (no Jev calls)
RUN_OFFLINE_BENCHMARK = False

def benchmark_assistance(cases=((0,"center_normal"),(1,"center_normal"),(2,"center_normal"),(0,"random"))):
    results = []
    for seed, schedule in cases:
        test_env = gym.make(ENV_ID, continuous=True, domain_randomize=False, max_episode_steps=MAX_STEPS)
        try:
            obs, _, total = reset_race(test_env, seed)
            rng = np.random.default_rng(seed)
            choice = "center_normal"
            for step in range(CAMERA_WARMUP, MAX_STEPS):
                if step % CONTROL_HORIZON == 0:
                    choice = str(rng.choice(list(ASSISTED))) if schedule == "random" else schedule
                vision = extract_visual_state(obs, read_speed(test_env))
                action = to_gym_action(assisted_control(vision, choice))
                assert np.isfinite(action).all() and action[1]*action[2] == 0
                obs, reward, terminated, truncated, _ = test_env.step(action)
                total += float(reward)
                if terminated or truncated:
                    break
            row = dict(seed=seed, schedule=schedule, steps=step+1, reward=round(total,2),
                       coverage=round(test_env.unwrapped.tile_visited_count/len(test_env.unwrapped.track),4),
                       terminated=bool(terminated), truncated=bool(truncated))
            results.append(row)
            print(row, flush=True)
        finally:
            test_env.close()
    return results

if RUN_OFFLINE_BENCHMARK:
    offline_results = benchmark_assistance()
else:
    print("Set RUN_OFFLINE_BENCHMARK=True to test the lane follower without API requests.")


## Interpretation
The benchmark uses fixed or random adjustments; it measures the assisted controller, not Jev intelligence. It runs to the frame limit without the live run’s no-progress watchdog. Direct Jev driving is experimental and has not been validated against the live API. Road masking assumes the default non-randomized colors. Missing road detections stop acceleration; they cannot guarantee recovery after leaving the road.

Local Gymnasium 1.3.0 offline results (3,000-frame cap):

| Seed | Adjustment schedule | Steps | Reward | Road coverage |
|---|---|---:|---:|---:|
| 0 | center_normal | 2518 | +748.20 | 100.00% |
| 1 | center_normal | 2200 | +780.00 | 100.00% |
| 2 | center_normal | 3000 | +664.18 | 96.42% |
| 0 | random | 2538 | +746.20 | 100.00% |
